# herring_10_15_per_file

HB2407 Herring 10-15 pipeline with per-raw-file processing and a pre-MVBS fan-in.

**Author:** <Your Name>
**Version:** 1.0

In [ ]:
# Uncomment the lines below to install required packages.
# %pip install git+https://github.com/BLayman-NOAA/AA-SI_Utils.git
# %pip install git+https://github.com/BLayman-NOAA/AA-SI_calibration.git
# %pip install echopype>=0.6.0
# %pip install git+https://github.com/BLayman-NOAA/AA-SI_ML.git
# %pip install git+https://github.com/BLayman-NOAA/AA-SI_Visualization.git

In [ ]:
from aa_si_utils.utils import initial_setup_and_validation
from aa_si_calibration.calibration import generate_standardized_cal_mapping
from aa_si_utils.utils import read_raw_files_to_stores
from aa_si_utils.utils import combine_raw_stores
from aa_si_calibration.calibration import extract_standardized_calibration_parameters
from echopype.calibrate import compute_Sv
from aa_si_utils.utils import compute_transducer_depth
from echopype.consolidate import add_depth
from aa_si_utils.utils import read_seafloor_line_evl
from aa_si_utils.utils import create_seafloor_mask
from aa_si_utils.utils import create_surface_mask
from aa_si_utils.utils import combine_masks
from aa_si_utils.utils import apply_mask_to_sv
from aa_si_utils.utils import concat_datasets
from aa_si_utils.utils import crop_range_samples
from aa_si_utils.utils import rechunk_dataset
from aa_si_ml.ml import remove_noise
from aa_si_visualization.echogram import plot_sv_echogram
from aa_si_utils.utils import mask_sparse_bins
from echopype.commongrid import compute_MVBS
from aa_si_ml.ml import compute_per_cell_statistics
from aa_si_ml.ml import reshape_data_for_ml
from aa_si_ml.ml import add_auxiliary_features
from aa_si_ml.ml import normalize_data
from aa_si_visualization.echogram import plot_flattened_data_echogram
from aa_si_ml.ml import run_hdbscan
from aa_si_ml.ml import embed_clustering_results
from aa_si_ml.ml import plot_clustering_report


In [ ]:
raw_input_folder = None  # TODO: set this value  # path
cal_input_folder = './HB2407_calibration_files'  # path
seabed_line_folder = 'gs://ggn-nmfs-aa-prod-1-data/HDD/Henry_B_Bigelow/HB2407/Auxiliary/EV_seabedlines'  # path
file_time_start = '2024-10-15T13:38:52'  # str
file_time_end = '2024-10-15T23:43:36'  # str
min_depth = 0  # float
max_depth = None  # TODO: set this value  # float
time_min = None  # TODO: set this value  # str
time_max = None  # TODO: set this value  # str
x_axis_units = 'datetime'  # str
y_axis_units = 'meters'  # str
calibration_outputs = 'calibration'  # str
raw_file_names = []  # list
ping_time_chunk = 10000  # int
seabed_line_max_gap_s = 600  # float
seabed_line_min_coverage = 0.99  # float
crop_trailing_all_nan = True  # bool
crop_max_range_m = None  # TODO: set this value  # float
noise_chunk_ping_time = 1000  # int
mvbs_chunk_ping_time = 1000  # int
echodata = None  # TODO: set this value  # echodata
seafloor_depth = None  # TODO: set this value  # dataarray

## Included: processing_lvl_0_calibration.yaml

Steps: initial_setup, gen_cal_mapping

### Step: `initial_setup`
**Op:** `initial_setup`

Validate the raw files in the input folder, create the calibration output directory if needed, and return the raw file paths and that directory.

**Parameters:**
- `raw_input_folder` path - Folder of .raw files, local or a remote URL such as gs://bucket/survey/raw. Remote folders are listed without downloading any data.

- `calibration_outputs` str - Subdirectory under the run's outputs folder where calibration artifacts are written, with JSON logs in its logs/ subfolder.
- `raw_file_names` list - Explicit file name list; empty means include all files in folder
- `clear_previous_json_logs` bool - Delete the calibration_flags.json left by a previous run before this one starts, so the flags reflect this run alone. Set false to let the flags accumulate across runs.

- `file_time_start` str - Inclusive ISO datetime, for example "2016-07-25T20:58"; keeps only raw files whose filename time (D{YYYYMMDD}-T{HHMMSS}) is at or after it. Unset keeps all files.
- `file_time_end` str - Optional inclusive ISO datetime upper bound for filename-encoded file times. See file_time_start.

In [ ]:
# --- Parameters ---
_initial_setup__raw_input_folder = '${inputs.raw_input_folder}'
_initial_setup__calibration_outputs = 'calibration'
_initial_setup__raw_file_names = []
_initial_setup__clear_previous_json_logs = True
_initial_setup__file_time_start = '2024-10-15T13:38:52'
_initial_setup__file_time_end = '2024-10-15T23:43:36'

_result = initial_setup_and_validation(
    raw_input_folder=_initial_setup__raw_input_folder,
    calibration_outputs_string=_initial_setup__calibration_outputs,
    raw_file_names=_initial_setup__raw_file_names,
    clear_previous_json_logs=_initial_setup__clear_previous_json_logs,
    file_time_start=_initial_setup__file_time_start,
    file_time_end=_initial_setup__file_time_end,
)
raw_file_paths = _result['raw_file_paths']
calibration_output_dir = _result['calibration_output_dir']

### Step: `gen_cal_mapping`
**Op:** `generate_standardized_cal_mapping`

Read manufacturer calibration files, convert them to per-channel standardized files, match each raw channel to its calibration data, and return a mapping dict and a calibration dict.

**Parameters:**
- `raw_input_folder` path - Folder of raw files whose channel configuration is scanned, local or a remote URL such as gs://bucket/survey/raw.

- `cal_input_folder` path - Folder of manufacturer calibration files (.cal or .xml), local or a remote URL.

- `output_base` path - Root directory the standardized calibration files, mapping files, and logs are written under. Normally wired to ${initial_setup.calibration_output_dir}.

- `cruise_id` str - Cruise identifier recorded in every generated single-channel calibration file (e.g. "HB1603"), so a standardized file stays traceable to the survey it came from.

- `record_author` str - Name recorded as the author of every generated single-channel calibration file. Not used in any calculation.

- `short_filenames` bool - Use compact names for the generated single-channel files and the keys in mapping_dict. Set false for fully descriptive names, which are easier to read and longer.

- `keep_unused` bool - Retain standardized files not matched to any channel
- `conflict_resolution` str - 'error' or 'interactive' when multiple cal files match a channel
- `verbose` bool - Print progress information during mapping; implementation-specific
- `file_time_start` str - Inclusive ISO datetime; only raw files whose filename time (D{YYYYMMDD}-T{HHMMSS}) is at or after it are scanned. Match the initial_setup window. Unset scans from the first file.
- `file_time_end` str - Optional inclusive ISO datetime upper bound for filename-encoded file times. See file_time_start.

In [ ]:
# --- Parameters ---
_gen_cal_mapping__raw_input_folder = '${inputs.raw_input_folder}'
_gen_cal_mapping__cal_input_folder = './HB2407_calibration_files'
_gen_cal_mapping__output_base = calibration_output_dir
_gen_cal_mapping__cruise_id = 'HB2407'
_gen_cal_mapping__record_author = '<Your Name>'
_gen_cal_mapping__short_filenames = True
_gen_cal_mapping__keep_unused = True
_gen_cal_mapping__conflict_resolution = 'error'
_gen_cal_mapping__file_time_start = '2024-10-15T13:38:52'
_gen_cal_mapping__file_time_end = '2024-10-15T23:43:36'

_result = generate_standardized_cal_mapping(
    raw_input_folder=_gen_cal_mapping__raw_input_folder,
    cal_input_folder=_gen_cal_mapping__cal_input_folder,
    output_base=_gen_cal_mapping__output_base,
    cruise_id=_gen_cal_mapping__cruise_id,
    record_author=_gen_cal_mapping__record_author,
    short_filenames=_gen_cal_mapping__short_filenames,
    keep_unused=_gen_cal_mapping__keep_unused,
    conflict_resolution=_gen_cal_mapping__conflict_resolution,
    file_time_start=_gen_cal_mapping__file_time_start,
    file_time_end=_gen_cal_mapping__file_time_end,
)
mapping_dict = _result['mapping_dict']
calibration_dict = _result['calibration_dict']

*End of included section: processing_lvl_0_calibration.yaml*

## Included: processing_lvl_1_per_file.yaml

Steps: read_raw, combine_raw

### Step: `read_raw`
**Op:** `read_raw_files`

Open raw echosounder files with echopype and write each one to an intermediate store. Returns the store paths, or in-memory EchoData objects when intermediate_format is "none".

**Parameters:**
- `sonar_model` str - Sonar model string recognized by echopype (e.g. 'EK60', 'EK80')
- `include_bot` bool - Include bottom-detection (.bot) data if available
- `intermediate_format` str - Store format written between reading and combining. "netcdf" (default, recommended) writes temporary .nc files. "none" keeps everything in memory. "zarr" writes zarr stores, which can live on a gs:// temp dir.

- `use_swap` str - Whether echopype swaps data to disk while reading. "auto" (default) swaps a file whose expanded data would exceed about 40% of total RAM.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `combine_raw`
**Op:** `combine_raw_files`

Combine the per-file stores or EchoData objects from read_raw_files into one EchoData object. Mark the step checkpoint: always so the combined output is written once and reused.

**Parameters:**
- `ping_time_chunk` int - Chunk length along ping_time for the combined EchoData and its checkpoint, in pings. Larger values write faster to gs:// but use more memory; try 2000 to 4000 for bucket caches.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `extract_cal_params`
**Op:** `extract_standardized_cal_params`

Extract calibration parameters (gain, sa-correction, EBA) and environmental parameters (sound speed, absorption) from standardized calibration files, cross-referenced against the EchoData object for channel alignment.

**Parameters:**
- `raw_file_path` path - Raw file whose mapping to use, matched by basename. Wire to ${_item} in a per-file chain. Unset uses the first file in the survey, which suits whole-survey recipes.

- `unmapped_channels` str - What to do with a channel no calibration file matched. "warn" (default) reports it and uses the values the raw file records. "error" stops the run.

- `output_logs_folder` path - Where the fallback writes its calibration flags. Unset uses a temporary folder.

- `waveform_mode` str - The waveform_mode compute_sv uses, "CW" or "BB". Selects which beam group's channels are extracted, for EK80 files mixing CW and FM. Unset uses the first beam group.

- `encode_mode` str - The encode_mode the compute_sv step passes to echopype, "complex" or "power". Set it together with waveform_mode, or leave both unset.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `compute_sv`
**Op:** `compute_sv`

Compute volume backscattering strength (Sv) from raw EchoData, applying cal_params and env_params when given and the raw file's own values otherwise.

*Runs once per `map_over` instance (fan-out loop).*

### Step: `compute_transducer_depth`
**Op:** `compute_transducer_depth`

Compute per-ping transducer depth in metres below the water surface from the EK Platform group, for feeding ``ep_add_depth`` as its ``depth_offset``. Use it instead of ``use_platform_vertical_offsets``, which gets the sign wrong.

**Parameters:**
- `datum_correction_m` float (m) - Metres added to the static offset when it is measured from a hull datum rather than the waterline. Positive pushes the transducer deeper. The default 0.0 takes the recorded offset as waterline referenced.

- `use_heave` bool - Add the per-ping ``vertical_offset`` (heave) term. False holds the static keel depth on every ping. Without usable heave the step says so and uses the static depth.

- `heave_sign` float - Sign applied to ``vertical_offset`` (1.0 or -1.0). The default was verified on HB2407 (EK80); check it on a new vessel or instrument setup.

- `interp_method` str - Interpolation used to put heave on ping_time, such as "linear" (default) or "nearest".


*Runs once per `map_over` instance (fan-out loop).*

### Step: `ep_add_depth`
**Op:** `ep_add_depth`

Add a ``depth`` variable, in metres below the surface, to an Sv Dataset using echopype. Existing variables, including ``echo_range``, are kept.

**Parameters:**
- `depth_offset` (m) - Transducer depth in metres, as a constant or a per-ping DataArray such as ``compute_transducer_depth``'s output. When set, takes precedence over Platform vertical offsets.

- `tilt` float (degree) - Constant transducer tilt; 0 corresponds to a vertically-pointing transducer. When set, takes precedence over Platform/Beam angles.

- `downward` bool - True when transducers point downward (typical hull-mounted).
- `use_platform_vertical_offsets` bool - Use Echodata Platform group vertical offsets to compute transducer depth. EK60/EK80 only. Ignored when ``depth_offset`` is set.

- `use_platform_angles` bool - Use Echodata Platform group angles to compute echo_range scaling. EK60/EK80 only. Cannot be combined with ``use_beam_angles``.

- `use_beam_angles` bool - Use Echodata Beam group angles to compute echo_range scaling. EK60/EK80 only. Cannot be combined with ``use_platform_angles``.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `detect_seafloor`
**Op:** `read_seafloor_line`

Read a seafloor line from Echoview ``.evl`` files and align it to the Sv pings, in metres. Swaps in for the seafloor detection ops. Pings the line does not cover are NaN and get masked entirely downstream.

**Parameters:**
- `evl_path` path - An Echoview ``.evl`` line file, or a folder of them selected with file_time_start and file_time_end. Local or remote (e.g. gs://...). Give exactly one of this and ``window``.

- `window` dict - Mapping with the line path under ``window_key``, used instead of ``evl_path`` in a mapped step (``window: ${_item}``). It can also supply file_time_start and file_time_end.

- `window_key` str - Key the line path is read from when ``window`` is given. Defaults to "seabed_evl", which is what plan_dive_datasets emits.

- `vertical_reference` str - What the EVL depths are measured from: "surface" (default, Echoview's usual) or "transducer" (range along the beam). The line is converted to ds_Sv's own reference.

- `min_status` int - Drop line points whose Echoview status is below this value (0 = none, 1 = unverified, 2 = bad, 3 = good). Null (default) keeps every point. Dropped points leave a gap, subject to max_gap_s.

- `max_gap_s` float (s) - Widest hole in the line to interpolate across; pings in a wider hole are NaN. Null (default) interpolates across holes of any width.

- `edge_extend_s` float (s) - Seconds past the line's first and last point to hold its end depths. 0.0 (default) leaves pings outside the line NaN; null holds the end depths indefinitely.

- `depth_offset_m` float (m) - Constant metres added to the line, for when the transducer draft configured in Echoview differs from the one baked into ds_Sv['depth']. Positive pushes the seafloor deeper.

- `min_coverage` float - Fraction of pings (0-1) that must end up with a finite seafloor depth; below it the step fails instead of silently masking the uncovered pings away. Default 0.0 never fails.

- `file_time_start` str - Optional inclusive ISO datetime (e.g. "2024-10-12T00:15") selecting line files in an evl_path folder by the times in their names. Use the window that selected the raw files. Ignored for a single file.

- `file_time_end` str - Optional inclusive ISO datetime upper bound for line file selection. See file_time_start.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `create_seafloor_mask`
**Op:** `create_seafloor_mask`

Build a seafloor mask from baseline Sv and detected seafloor depth. The seafloor depth is passed as an explicit input, enabling swappable detection methods without modifying masking logic.

**Parameters:**
- `seafloor_buffer_m` float (m) - Height above the seafloor line to also mask, covering detection error and the near-bottom dead zone at the cost of some real backscatter.

- `range_var` str - Meter-valued variable on ds_Sv compared against the seafloor line, on the same vertical reference. Null (default) uses "depth" if present, otherwise "echo_range".


*Runs once per `map_over` instance (fan-out loop).*

### Step: `create_surface_mask`
**Op:** `create_surface_mask`

Build a surface-interference mask from baseline Sv using a selectable method.

**Parameters:**
- `surface_depth_m` float (m) - Depth above which samples are masked out, removing the transmit pulse ringdown and near-surface bubble layer. The default of 0.0 masks nothing.


*Runs once per `map_over` instance (fan-out loop).*

### Step: `combine_masks`
**Op:** `combine_masks`

Combine two or more mask datasets into one final mask. Accepts a list of mask inputs to support inserting additional custom mask-producing steps into the chain without changing core framework code.

**Parameters:**
- `mode` str - Combination mode: 'and' (all must be valid) or 'or' (any must be valid)

*Runs once per `map_over` instance (fan-out loop).*

### Step: `apply_mask`
**Op:** `apply_sv_mask`

Apply a boolean mask to an Sv Dataset, setting masked samples to NaN.

*Runs once per `map_over` instance (fan-out loop).*

In [ ]:
raw_stores = []
echodata = []
cal_params = []
env_params = []
other_params = []
compute_sv_ds_Sv = []
transducer_depth = []
ep_add_depth_ds_Sv = []
seafloor_depth = []
create_seafloor_mask_mask = []
create_surface_mask_mask = []
combine_masks_mask = []
apply_mask_ds_Sv = []
for _item in raw_file_paths:
    _iter_raw_stores = read_raw_files_to_stores(
        raw_file_paths=[_item],
        sonar_model='EK80',
        include_bot=False,
        intermediate_format='zarr',
    )
    raw_stores.append(_iter_raw_stores)
    _iter_echodata = combine_raw_stores(raw_stores=_iter_raw_stores, ping_time_chunk=10000)
    echodata.append(_iter_echodata)
    _result = extract_standardized_calibration_parameters(
        calibration_dict=calibration_dict,
        mapping_dict=mapping_dict,
        echodata=_iter_echodata,
        raw_file_path=_item,
    )
    _iter_cal_params = _result['cal_params']
    _iter_env_params = _result['env_params']
    _iter_other_params = _result['other_params']
    cal_params.append(_iter_cal_params)
    env_params.append(_iter_env_params)
    other_params.append(_iter_other_params)
    _iter_compute_sv_ds_Sv = compute_Sv(
        echodata=_iter_echodata,
        cal_params=_iter_cal_params,
        env_params=_iter_env_params,
        waveform_mode='CW',
        encode_mode='power',
    )
    compute_sv_ds_Sv.append(_iter_compute_sv_ds_Sv)
    _iter_transducer_depth = compute_transducer_depth(
        ds_Sv=_iter_compute_sv_ds_Sv,
        echodata=_iter_echodata,
        datum_correction_m=-1.0,
        use_heave=True,
        heave_sign=1.0,
    )
    transducer_depth.append(_iter_transducer_depth)
    _iter_ep_add_depth_ds_Sv = add_depth(
        ds=_iter_compute_sv_ds_Sv,
        echodata=_iter_echodata,
        depth_offset=_iter_transducer_depth,
    )
    ep_add_depth_ds_Sv.append(_iter_ep_add_depth_ds_Sv)
    _iter_seafloor_depth = read_seafloor_line_evl(
        ds_Sv=_iter_ep_add_depth_ds_Sv,
        evl_path='gs://ggn-nmfs-aa-prod-1-data/HDD/Henry_B_Bigelow/HB2407/Auxiliary/EV_seabedlines',
        file_time_start='2024-10-15T13:38:52',
        file_time_end='2024-10-15T23:43:36',
        edge_extend_s=0.0,
        max_gap_s=600,
        min_coverage=0.99,
    )
    seafloor_depth.append(_iter_seafloor_depth)
    _iter_create_seafloor_mask_mask = create_seafloor_mask(
        ds_Sv=_iter_ep_add_depth_ds_Sv,
        seafloor_depth=_iter_seafloor_depth,
        seafloor_buffer_m=1,
    )
    create_seafloor_mask_mask.append(_iter_create_seafloor_mask_mask)
    _iter_create_surface_mask_mask = create_surface_mask(ds_Sv=_iter_compute_sv_ds_Sv, surface_depth_m=5.0)
    create_surface_mask_mask.append(_iter_create_surface_mask_mask)
    _iter_combine_masks_mask = combine_masks(masks=[_iter_create_seafloor_mask_mask, _iter_create_surface_mask_mask], mode='and')
    combine_masks_mask.append(_iter_combine_masks_mask)
    _iter_apply_mask_ds_Sv = apply_mask_to_sv(ds_Sv=_iter_ep_add_depth_ds_Sv, mask=_iter_combine_masks_mask)
    apply_mask_ds_Sv.append(_iter_apply_mask_ds_Sv)

*End of included section: processing_lvl_1_per_file.yaml*

## Included: processing_lvl_2_per_file.yaml

Steps: extract_cal_params, compute_sv, compute_transducer_depth, ep_add_depth, detect_seafloor, create_seafloor_mask, create_surface_mask, combine_masks, apply_mask, merge_sv, crop_range, rechunk_before_noise, remove_noise

### Step: `merge_sv`
**Op:** `merge_datasets`

Concatenate a collected list of Datasets into one along ``dim``, for example per-file MVBS gathered with ``collect``. A single Dataset passes through unchanged.

**Parameters:**
- `dim` str - Dimension to concatenate the collected Datasets along.
- `on_duplicate` str - How to handle two collected segments with the same label on ``dim``. Unset keeps both (fine when segments can't overlap). "mean" averages them (binned products split at a segment boundary). "first" keeps the first and sorts (overlapping slices of one store).


In [ ]:
# --- Parameters ---
_merge_sv__dim = 'ping_time'

merged = concat_datasets(datasets=apply_mask_ds_Sv, dim=_merge_sv__dim)

### Step: `crop_range`
**Op:** `crop_range`

Trim the deep range samples that hold no data after seafloor masking. Run it on the whole survey, after the per-file fan-in. Active criteria each give a cut and the shallowest wins.

**Parameters:**
- `max_range_m` float (m) - Hard cap on range. Samples deeper than this are dropped whether or not they hold data. Unset derives the crop from the data.

- `outlier_sigma` float - Standard deviations above the mean of each ping's deepest finite sample at which a ping counts as an outlier and is ignored. Unset disables outlier rejection and uses drop_trailing_all_nan.

- `outlier_margin` float - Multiplier on (mean + outlier_sigma x sd) giving the outlier threshold. Raise it to flag fewer pings. Only used when outlier_sigma is set.

- `drop_trailing_all_nan` bool - Crop to one past the deepest sample with a finite value anywhere (lossless). Ignored when outlier_sigma is set. False, with the other two unset, turns the step into a no-op.

- `data_var` str - Variable inspected for finite values.
- `range_var` str - Variable supplying each sample's range in metres.

In [ ]:
# --- Parameters ---
_crop_range__outlier_sigma = 3.0
_crop_range__outlier_margin = 1.5
_crop_range__drop_trailing_all_nan = True
_crop_range__max_range_m = crop_max_range_m

crop_range_ds_Sv = crop_range_samples(
    ds_Sv=merged,
    outlier_sigma=_crop_range__outlier_sigma,
    outlier_margin=_crop_range__outlier_margin,
    drop_trailing_all_nan=_crop_range__drop_trailing_all_nan,
    max_range_m=_crop_range__max_range_m,
)

### Step: `rechunk_before_noise`
**Op:** `rechunk_dataset`

Rechunk a Dataset along ping_time so the steps after it can run dask-parallel.

**Parameters:**
- `ping_time_chunk` int - Target chunk size along ping_time.

In [ ]:
# --- Parameters ---
_rechunk_before_noise__ping_time_chunk = 1000

rechunk_before_noise_ds_Sv = rechunk_dataset(ds_Sv=crop_range_ds_Sv, ping_time_chunk=_rechunk_before_noise__ping_time_chunk)

### Step: `remove_noise`
**Op:** `remove_background_noise`

Estimate and remove background noise from Sv using the De Robertis and Higginbottom method. Returns a cleaned Sv Dataset whose Sv holds the corrected values.

**Parameters:**
- `ping_num` int - Number of pings used to estimate the noise level
- `range_sample_num` int - Number of range samples used to estimate the noise level
- `background_noise_max` str - Upper limit on the estimated noise level (e.g. '-125dB'); omit to leave it unbounded
- `SNR_threshold` str - Signal-to-noise ratio below which corrected samples are masked out (e.g. '3.0dB')

In [ ]:
# --- Parameters ---
_remove_noise__range_sample_num = 10
_remove_noise__ping_num = 5

remove_noise_ds_Sv = remove_noise(
    ds_Sv=rechunk_before_noise_ds_Sv,
    range_sample_num=_remove_noise__range_sample_num,
    ping_num=_remove_noise__ping_num,
)

*End of included section: processing_lvl_2_per_file.yaml*

## Step: `plot_sv_clean`
**Op:** `plot_sv_echogram`

Render an Sv echogram of ds_Sv, optionally on the coordinate axes of ds_Sv_source (for example MVBS on the original Sv grid). A sink step: it makes a figure and has no data output.

**Parameters:**
- `min_depth` float - Minimum depth (y-axis) in meters
- `max_depth` float - Maximum depth (y-axis) in meters. Omit to use the deepest depth present in the data.

- `ping_min` int - First ping index to display
- `ping_max` int - Last ping index to display. Omit to display through the last ping.
- `time_min` str - Start of the displayed window as a UTC timestamp, for example "2024-10-15T12:30". Resolved to the closest ping and used instead of ping_min. Omit to window by ping index.

- `time_max` str - End of the displayed window as a UTC timestamp. Resolved to the closest ping and used instead of ping_max. Omit to window by ping index.

- `x_axis_units` str - Units for the x-axis: 'datetime' (default, absolute UTC clock times), 'seconds' from the first ping shown, 'pings', 'meters' (distance travelled, derived from the GPS track), or 'bins' (MVBS input only).

- `y_axis_units` str - Units for the y-axis: 'meters' (default), 'range_sample', or 'bins' (MVBS input only).

- `panel_width_to_height` float - Force each frequency panel to be this many times wider than tall (for example 8 for a long strip). Omit for automatic sizing.

- `overlay_lines` list - Optional list of line overlays. Each entry is a dict with 'var' (variable name in ds_Sv) and 'style' (dict of matplotlib kwargs).

- `save_image` bool - Save the rendered figure to an image file; null uses runtime defaults
- `save_formats` list - Image formats to save, chosen from png, svg, jpeg, jpg, and pdf
- `save_dir` str - Optional directory override for saved image artifacts
- `show` bool - Whether to display the figure interactively or inline; null uses runtime defaults
- `dpi` int - Optional DPI override for raster image outputs

In [ ]:
# --- Parameters ---
_plot_sv_clean__min_depth = 0
_plot_sv_clean__max_depth = max_depth
_plot_sv_clean__time_min = time_min
_plot_sv_clean__time_max = time_max
_plot_sv_clean__x_axis_units = 'datetime'
_plot_sv_clean__y_axis_units = 'meters'
_plot_sv_clean__dpi = 100

plot_sv_echogram(
    ds_Sv=remove_noise_ds_Sv,
    min_depth=_plot_sv_clean__min_depth,
    max_depth=_plot_sv_clean__max_depth,
    time_min=_plot_sv_clean__time_min,
    time_max=_plot_sv_clean__time_max,
    x_axis_units=_plot_sv_clean__x_axis_units,
    y_axis_units=_plot_sv_clean__y_axis_units,
    dpi=_plot_sv_clean__dpi,
)

## Included: processing_lvl_3.yaml

Steps: mask_sparse, rechunk_before_mvbs, compute_mvbs, compute_cell_stats

### Step: `mask_sparse`
**Op:** `mask_sparse_bins`

Remove spatiotemporal bins with fewer valid samples than a threshold, reducing noise in MVBS computation.

**Parameters:**
- `range_bin` str - Range bin size (e.g. '2m')
- `ping_time_bin` str - Ping-time bin size (e.g. '10s')
- `nan_threshold` float - Fraction of NaN values above which a bin is masked (0.0 to 1.0)
- `range_var` str - Meter-valued variable to bin along: 'echo_range' (from the transducer face) or 'depth' (from the surface; the dataset must carry depth). Must match compute_mvbs and compute_per_cell_statistics.


In [ ]:
# --- Parameters ---
_mask_sparse__range_bin = '4m'
_mask_sparse__ping_time_bin = '4s'
_mask_sparse__nan_threshold = 0.95

mask_sparse_ds_Sv = mask_sparse_bins(
    ds_Sv=remove_noise_ds_Sv,
    range_bin=_mask_sparse__range_bin,
    ping_time_bin=_mask_sparse__ping_time_bin,
    nan_threshold=_mask_sparse__nan_threshold,
)

### Step: `rechunk_before_mvbs`
**Op:** `rechunk_dataset`

Rechunk a Dataset along ping_time so the steps after it can run dask-parallel.

**Parameters:**
- `ping_time_chunk` int - Target chunk size along ping_time.

In [ ]:
# --- Parameters ---
_rechunk_before_mvbs__ping_time_chunk = 1000

rechunk_before_mvbs_ds_Sv = rechunk_dataset(ds_Sv=mask_sparse_ds_Sv, ping_time_chunk=_rechunk_before_mvbs__ping_time_chunk)

### Step: `compute_mvbs`
**Op:** `compute_mvbs`

Compute Mean Volume Backscattering Strength by averaging Sv over range and time bins. Returns a gridded MVBS Dataset.

**Parameters:**
- `range_bin` str (m) - Depth bin size for averaging (e.g. '2m')
- `ping_time_bin` str (s) - Ping-time bin size for averaging (e.g. '10s')
- `range_var` str - Meter-valued variable to bin along: 'echo_range' (from the transducer face) or 'depth' (from the surface; the dataset must carry depth, which ep_add_depth adds).

- `range_var_max` str (m) - Deepest value the bins cover, as a unit string (e.g. '2100m'). Unset reads the maximum from the data; supplying it keeps the step lazy.

- `closed` str - Which side of each bin interval is closed: 'left' or 'right'
- `skipna` bool - Skip NaN values when averaging. When false, any NaN in a bin makes the whole bin NaN, so masked data collapses.

- `fill_value` float - Value for bins holding no data. Unset keeps echopype's default, NaN.

- `method` str - Flox reduction strategy for dask arrays, normally 'map-reduce'. Any other value needs reindex set to null in the same params block.

- `reindex` bool - Reindex at the blockwise stage rather than after it. False uses less memory and more time. Only meaningful when method is 'map-reduce'.


In [ ]:
# --- Parameters ---
_compute_mvbs__range_bin = '4m'
_compute_mvbs__ping_time_bin = '4s'

ds_MVBS = compute_MVBS(
    ds_Sv=rechunk_before_mvbs_ds_Sv,
    range_bin=_compute_mvbs__range_bin,
    ping_time_bin=_compute_mvbs__ping_time_bin,
)

### Step: `compute_cell_stats`
**Op:** `compute_per_cell_statistics`

Compute statistics within each MVBS cell (such as the coefficient of variation) from fine-resolution Sv, at MVBS bin resolution. Use the same bins as mask_sparse_bins and compute_mvbs, and pass the result to add_auxiliary_features as ds_sv.

**Parameters:**
- `range_bin` str - Range bin size. Must match mask_sparse_bins and compute_mvbs (e.g. '2m')
- `ping_time_bin` str - Ping-time bin size. Must match mask_sparse_bins and compute_mvbs (e.g. '10s')
- `statistics` list - List of statistics to compute. Supported: ['cv']
- `data_var` str - Name of the Sv variable in the input Dataset
- `range_var` str - Meter-valued variable to bin along: 'echo_range' (from the transducer face) or 'depth' (from the surface; the dataset must carry depth). Must match mask_sparse_bins and compute_mvbs.

- `range_var_max` str (m) - Deepest value the cells cover, as a unit string (e.g. '1810m'), matching compute_mvbs. Unset reads the extent from the data. Pin it whenever the results will be merged.

- `cells_only` bool - Return only the statistic arrays instead of a copy of ds_Sv carrying them. Set true whenever this step is checkpointed or its results are merged.


In [ ]:
# --- Parameters ---
_compute_cell_stats__range_bin = '4m'
_compute_cell_stats__ping_time_bin = '4s'
_compute_cell_stats__statistics = ['cv']

compute_cell_stats_ds_Sv = compute_per_cell_statistics(
    ds_Sv=remove_noise_ds_Sv,
    range_bin=_compute_cell_stats__range_bin,
    ping_time_bin=_compute_cell_stats__ping_time_bin,
    statistics=_compute_cell_stats__statistics,
)

*End of included section: processing_lvl_3.yaml*

## Step: `plot_mvbs`
**Op:** `plot_sv_echogram`

Render an Sv echogram of ds_Sv, optionally on the coordinate axes of ds_Sv_source (for example MVBS on the original Sv grid). A sink step: it makes a figure and has no data output.

**Parameters:**
- `min_depth` float - Minimum depth (y-axis) in meters
- `max_depth` float - Maximum depth (y-axis) in meters. Omit to use the deepest depth present in the data.

- `ping_min` int - First ping index to display
- `ping_max` int - Last ping index to display. Omit to display through the last ping.
- `time_min` str - Start of the displayed window as a UTC timestamp, for example "2024-10-15T12:30". Resolved to the closest ping and used instead of ping_min. Omit to window by ping index.

- `time_max` str - End of the displayed window as a UTC timestamp. Resolved to the closest ping and used instead of ping_max. Omit to window by ping index.

- `x_axis_units` str - Units for the x-axis: 'datetime' (default, absolute UTC clock times), 'seconds' from the first ping shown, 'pings', 'meters' (distance travelled, derived from the GPS track), or 'bins' (MVBS input only).

- `y_axis_units` str - Units for the y-axis: 'meters' (default), 'range_sample', or 'bins' (MVBS input only).

- `panel_width_to_height` float - Force each frequency panel to be this many times wider than tall (for example 8 for a long strip). Omit for automatic sizing.

- `overlay_lines` list - Optional list of line overlays. Each entry is a dict with 'var' (variable name in ds_Sv) and 'style' (dict of matplotlib kwargs).

- `save_image` bool - Save the rendered figure to an image file; null uses runtime defaults
- `save_formats` list - Image formats to save, chosen from png, svg, jpeg, jpg, and pdf
- `save_dir` str - Optional directory override for saved image artifacts
- `show` bool - Whether to display the figure interactively or inline; null uses runtime defaults
- `dpi` int - Optional DPI override for raster image outputs

In [ ]:
# --- Parameters ---
_plot_mvbs__min_depth = 0
_plot_mvbs__max_depth = max_depth
_plot_mvbs__time_min = time_min
_plot_mvbs__time_max = time_max
_plot_mvbs__x_axis_units = 'datetime'
_plot_mvbs__y_axis_units = 'meters'
_plot_mvbs__dpi = 50

plot_sv_echogram(
    ds_Sv=ds_MVBS,
    ds_Sv_original=mask_sparse_ds_Sv,
    min_depth=_plot_mvbs__min_depth,
    max_depth=_plot_mvbs__max_depth,
    time_min=_plot_mvbs__time_min,
    time_max=_plot_mvbs__time_max,
    x_axis_units=_plot_mvbs__x_axis_units,
    y_axis_units=_plot_mvbs__y_axis_units,
    dpi=_plot_mvbs__dpi,
)

## Included: processing_lvl_4.yaml

Steps: reshape_ml, add_aux_features, normalize_ml, plot_normalized_ml, run_hdbscan, embed_results, plot_clustering_report

### Step: `reshape_ml`
**Op:** `reshape_for_ml`

Reshape an MVBS Dataset into a flat, ML-ready xarray Dataset where each observation corresponds to one spatial or temporal bin.

**Parameters:**
- `data_var` str - Name of the data variable to use as features
- `dataset_name` str - Name this ML dataset is stored under inside the returned Dataset. Every later ML step takes the same dataset_name to find it, so a recipe can carry several ML datasets side by side.

- `feature_strategy` str - Feature construction strategy (e.g. 'mean_centered')
- `baseline_channel` int - Channel index used as the baseline when computing relative features

In [ ]:
# --- Parameters ---
_reshape_ml__data_var = 'Sv'
_reshape_ml__dataset_name = 'ml_dataset'
_reshape_ml__feature_strategy = 'mean_centered'
_reshape_ml__baseline_channel = 0

reshape_ml_ds_ml_ready = reshape_data_for_ml(
    ds_Sv=ds_MVBS,
    data_var=_reshape_ml__data_var,
    dataset_name=_reshape_ml__dataset_name,
    feature_strategy=_reshape_ml__feature_strategy,
    baseline_channel=_reshape_ml__baseline_channel,
)

### Step: `add_aux_features`
**Op:** `add_auxiliary_features`

Append auxiliary coordinate-derived features (e.g. depth, ping_time_seconds) to the flattened ML-ready Dataset. Optional step; omit when no auxiliary features are needed.

**Parameters:**
- `dataset_name` str - ML dataset to append the features to. Must match the dataset_name used by reshape_for_ml.

- `features` list - List of built-in feature name strings to append. Supported built-ins: 'depth', 'ping_time_seconds', 'seafloor_depth', 'altitude', 'cell_cv'. 'seafloor_depth' and 'altitude' require the seafloor_depth or echodata input. 'cell_cv' requires the ds_sv input (from compute_per_cell_statistics).


In [ ]:
# --- Parameters ---
_add_aux_features__dataset_name = 'ml_dataset'
_add_aux_features__features = ['cell_cv']

add_aux_features_ds_ml_ready = add_auxiliary_features(
    ds_ml_ready=reshape_ml_ds_ml_ready,
    echodata=echodata,
    ds_sv=compute_cell_stats_ds_Sv,
    seafloor_depth=seafloor_depth,
    dataset_name=_add_aux_features__dataset_name,
    features=_add_aux_features__features,
)

### Step: `normalize_ml`
**Op:** `normalize_ml_data`

Normalize ML features in the flattened Dataset. Supports per-feature normalization methods via per_group_methods.

**Parameters:**
- `method` str - Global normalization method (e.g. 'flatten', 'minmax', 'zscore')
- `dataset_name` str - ML dataset to normalize. Must match the dataset_name used by reshape_for_ml.

- `normalization_name` str - Name this normalized result is stored under. run_hdbscan, the ML plots, and filter_normalized_by_cluster take the same name to find it, so several normalizations of one dataset can coexist and be compared.

- `shift_positive` bool - Shift all values to be non-negative after normalization
- `feature_weights` list - Per-feature weight multipliers applied before normalization
- `per_group_methods` dict - Map of feature name to individual normalization method override

In [ ]:
# --- Parameters ---
_normalize_ml__method = 'flatten'
_normalize_ml__dataset_name = 'ml_dataset'
_normalize_ml__normalization_name = 'normalized_flatten_mean_centered'
_normalize_ml__shift_positive = False

normalize_ml_ds_normalized = normalize_data(
    ds_ml_ready=add_aux_features_ds_ml_ready,
    method=_normalize_ml__method,
    dataset_name=_normalize_ml__dataset_name,
    normalization_name=_normalize_ml__normalization_name,
    shift_positive=_normalize_ml__shift_positive,
)

### Step: `plot_normalized_ml`
**Op:** `plot_ml_echogram`

Render a normalized ML feature echogram for inspection. This is a sink step: it produces a side-effect (figure) with no data output.

**Parameters:**
- `dataset_name` str - ML dataset to plot. Must match the dataset_name used by reshape_for_ml.

- `normalization_name` str - Normalized feature set to plot. Must match the normalization_name used by normalize_ml_data.

- `min_depth` float - Minimum depth (y-axis) in meters
- `max_depth` float - Maximum depth (y-axis) in meters. Omit to use the deepest depth present in the data.

- `ping_min` int - First ping index to display
- `ping_max` int - Last ping index to display. Omit to display through the last ping.
- `time_min` str - Start of the displayed window as a UTC timestamp, for example "2024-10-15T12:30". Resolved to the closest ping and used instead of ping_min. Omit to window by ping index.

- `time_max` str - End of the displayed window as a UTC timestamp. Resolved to the closest ping and used instead of ping_max. Omit to window by ping index.

- `x_axis_units` str - Units for the x-axis: 'datetime' (default, absolute UTC clock times), 'seconds' from the first ping shown, 'pings', 'meters' (distance travelled, derived from the GPS track), or 'bins' (MVBS-derived data only).

- `y_axis_units` str - Units for the y-axis: 'meters' (default), 'range_sample', or 'bins' (MVBS-derived data only).

- `save_image` bool - Save the rendered figure to an image file; null uses runtime defaults
- `save_formats` list - Image formats to save, chosen from png, svg, jpeg, jpg, and pdf
- `save_dir` str - Optional directory override for saved image artifacts
- `show` bool - Whether to display the figure interactively or inline; null uses runtime defaults
- `dpi` int - Optional DPI override for raster image outputs

In [ ]:
# --- Parameters ---
_plot_normalized_ml__dataset_name = 'ml_dataset'
_plot_normalized_ml__normalization_name = 'normalized_flatten_mean_centered'
_plot_normalized_ml__min_depth = 0
_plot_normalized_ml__max_depth = max_depth
_plot_normalized_ml__time_min = time_min
_plot_normalized_ml__time_max = time_max
_plot_normalized_ml__x_axis_units = 'datetime'
_plot_normalized_ml__y_axis_units = 'meters'
_plot_normalized_ml__dpi = 30

plot_flattened_data_echogram(
    ds_ml=normalize_ml_ds_normalized,
    ds_Sv_original=remove_noise_ds_Sv,
    ml_dataset_name=_plot_normalized_ml__dataset_name,
    ml_specific_data_name=_plot_normalized_ml__normalization_name,
    min_depth=_plot_normalized_ml__min_depth,
    max_depth=_plot_normalized_ml__max_depth,
    time_min=_plot_normalized_ml__time_min,
    time_max=_plot_normalized_ml__time_max,
    x_axis_units=_plot_normalized_ml__x_axis_units,
    y_axis_units=_plot_normalized_ml__y_axis_units,
    dpi=_plot_normalized_ml__dpi,
)

### Step: `run_hdbscan`
**Op:** `run_hdbscan`

Run HDBSCAN unsupervised clustering on normalized feature data. A good sweep target for comparing hyperparameters.

**Parameters:**
- `dataset_name` str - ML dataset to cluster. Must match the dataset_name used by reshape_for_ml.

- `normalization_name` str - Normalized feature set to cluster. Must match the normalization_name used by normalize_ml_data.

- `ml_result_name` str - Name this clustering result is stored under. embed_clustering_results and plot_clustering_report take the same name to find it, so a recipe can keep several clusterings of one dataset side by side.

- `min_cluster_size` int - Absolute minimum cluster size. Mutually exclusive with min_cluster_size_fraction.
- `min_cluster_size_fraction` float - Minimum cluster size as a fraction of the points fitted, used when min_cluster_size is omitted. Mutually exclusive with min_cluster_size. Unset uses 0.03.

- `min_samples` int - Neighbourhood size HDBSCAN uses to judge how dense a point is. Higher values make the run more conservative, pushing sparse points into noise instead of into a cluster.

- `sample_size` int - Cap on how many samples are clustered. A larger dataset is clustered on a random subset of this size and the rest get no label (see assign_clusters_to_all_points). Set it above the sample count to cluster everything.

- `cluster_selection_method` str - HDBSCAN cluster selection method ('eom' or 'leaf')
- `use_hdbscan` bool - Use HDBSCAN; set false to fall back to a simpler clustering method
- `epsilon` float - DBSCAN epsilon distance parameter
- `find_background_cluster` bool - Run background-cluster detection path instead of standard HDBSCAN. When true, background_label output is populated.

- `soft_membership_threshold` float - Reassign noise points via soft-membership probabilities when set. null disables soft-membership reassignment.


In [ ]:
# --- Parameters ---
_run_hdbscan__dataset_name = 'ml_dataset'
_run_hdbscan__normalization_name = 'normalized_flatten_mean_centered'
_run_hdbscan__ml_result_name = 'hdbscan_results'
_run_hdbscan__min_cluster_size_fraction = 0.001
_run_hdbscan__min_samples = 7
_run_hdbscan__sample_size = 1000000
_run_hdbscan__cluster_selection_method = 'eom'
_run_hdbscan__use_hdbscan = True
_run_hdbscan__soft_membership_threshold = 0.6

_result = run_hdbscan(
    ds_normalized=normalize_ml_ds_normalized,
    dataset_name=_run_hdbscan__dataset_name,
    normalization_name=_run_hdbscan__normalization_name,
    ml_result_name=_run_hdbscan__ml_result_name,
    min_cluster_size_fraction=_run_hdbscan__min_cluster_size_fraction,
    min_samples=_run_hdbscan__min_samples,
    sample_size=_run_hdbscan__sample_size,
    cluster_selection_method=_run_hdbscan__cluster_selection_method,
    use_hdbscan=_run_hdbscan__use_hdbscan,
    soft_membership_threshold=_run_hdbscan__soft_membership_threshold,
)
clustering_results = _result['clustering_results']
clustering_model = _result['clustering_model']
background_label = _result['background_label']
cluster_labels = _result['cluster_labels']

### Step: `embed_results`
**Op:** `embed_clustering_results`

Embed clustering results into the normalized ML Dataset as cluster-label variables. A collected list of results embeds one variable per result, named from ml_result_name.

**Parameters:**
- `dataset_name` str - ML dataset the labels are embedded into. Must match the dataset_name used by reshape_for_ml.

- `ml_result_name` str - Name the embedded cluster labels are stored under, normally the ml_result_name from run_hdbscan so plot_clustering_report finds it. For a collected list, the stem of the per-result variable names.


In [ ]:
# --- Parameters ---
_embed_results__dataset_name = 'ml_dataset'
_embed_results__ml_result_name = 'hdbscan_results'

_result = embed_clustering_results(
    ds_normalized=normalize_ml_ds_normalized,
    clustering_results=clustering_results,
    dataset_name=_embed_results__dataset_name,
    ml_result_name=_embed_results__ml_result_name,
)
embed_results_ds_normalized = _result['ds_normalized']
gridded_results = _result['gridded_results']

### Step: `plot_clustering_report`
**Op:** `plot_clustering_report`

Render a clustering report: the clustering echogram, per-cluster statistics, and the HDBSCAN hierarchy plot when a fitted model is available. Produces figures only.

**Parameters:**
- `dataset_name` str - ML dataset to report on. Must match the dataset_name used by reshape_for_ml.

- `ml_result_name` str - Clustering result to report on. Must match the ml_result_name used by embed_clustering_results, which is the name the labels were stored under in ds_normalized.

- `plot_window` list - [min_depth, max_depth, ping_min, ping_max] for echogram display
- `time_min` str - Start of the displayed window as a UTC timestamp, for example "2024-10-15T12:30". Resolved to the closest ping and used instead of the ping_min entry in plot_window. Omit to window by ping index.

- `time_max` str - End of the displayed window as a UTC timestamp. Resolved to the closest ping and used instead of the ping_max entry in plot_window. Omit to window by ping index.

- `x_axis_units` str - Units for the clustering echogram x-axis: 'datetime' (default, absolute UTC clock times), 'seconds' from the first ping shown, 'pings', 'bins', or 'meters'.

- `y_axis_units` str - Units for the clustering echogram y-axis ('meters', 'range_sample', or 'bins')
- `overlay_line_var` str - Name of a variable in ds_normalized to draw as a line overlay
- `overlay_windows` list - Window dicts (as plan_dive_datasets emits) whose line files are drawn onto the echogram. Empty draws nothing.

- `overlay_window_keys` list - Window keys naming the line files to draw. Defaults to the three dive profile keys dive_fit_evl, dive_u99_evl and dive_l99_evl. A window missing a key is skipped for that key.

- `overlay_fit_style` dict - Matplotlib line keywords for the fitted dive profile, for example {color: "#23FFFF", linewidth: 3.0}, merged over the built-in solid cyan. Empty uses the built-in style.

- `overlay_bound_style` dict - The same for both 99% confidence bound lines, for example {color: "#D08BFF", linestyle: ":"}. Empty uses the built-in dotted violet, thinner than the fitted profile.

- `normalization_name` str - Normalized feature set to plot statistics for when cluster_stats_sv_data_var is "ml_features" and no clustering_results input is wired. Ignored otherwise.

- `cluster_colors` list - Hex color strings for each cluster label
- `y_to_x_aspect_ratio_override` float - Override the echogram aspect ratio; null uses the default
- `cluster_stats_sv_data_var` str - Sv variable name used when computing per-cluster statistics, or "ml_features" to plot the flattened ML features used for clustering
- `cluster_stats_compute_pairwise_differences` bool - Compute inter-channel diffs in cluster statistics, taken against the second-lowest-frequency channel

- `cluster_stats_dpi` int - DPI for the cluster statistics figure, which holds much smaller text than the echogram. Null falls back to dpi.

- `save_image` bool - Save the rendered figures to image files; null uses runtime defaults
- `save_formats` list - Image formats to save, chosen from png, svg, jpeg, jpg, and pdf
- `save_dir` str - Optional directory override for saved image artifacts
- `show` bool - Whether to display figures interactively or inline; null uses runtime defaults
- `dpi` int - Optional DPI override for raster image outputs

In [ ]:
# --- Parameters ---
_plot_clustering_report__dataset_name = 'ml_dataset'
_plot_clustering_report__ml_result_name = 'hdbscan_results'
_plot_clustering_report__plot_window = [0, '${inputs.max_depth}']
_plot_clustering_report__time_min = time_min
_plot_clustering_report__time_max = time_max
_plot_clustering_report__x_axis_units = 'datetime'
_plot_clustering_report__y_axis_units = 'meters'
_plot_clustering_report__dpi = 50
_plot_clustering_report__cluster_stats_dpi = 200
_plot_clustering_report__overlay_line_var = 'sw_dive_profile'
_plot_clustering_report__cluster_colors = ['#9D00FF', '#35E200', '#FF0000', '#2F00FF', '#FF8800', '#FFFB1C', '#FF44E9', '#00BE8BCA', '#00FFE5', '#3D8D00FF', '#9C67FFFF']
_plot_clustering_report__cluster_stats_sv_data_var = 'Sv'

plot_clustering_report(
    ds_normalized=embed_results_ds_normalized,
    clustering_results=clustering_results,
    clustering_model=clustering_model,
    ds_Sv=remove_noise_ds_Sv,
    dataset_name=_plot_clustering_report__dataset_name,
    ml_result_name=_plot_clustering_report__ml_result_name,
    plot_window=_plot_clustering_report__plot_window,
    time_min=_plot_clustering_report__time_min,
    time_max=_plot_clustering_report__time_max,
    x_axis_units=_plot_clustering_report__x_axis_units,
    y_axis_units=_plot_clustering_report__y_axis_units,
    dpi=_plot_clustering_report__dpi,
    cluster_stats_dpi=_plot_clustering_report__cluster_stats_dpi,
    overlay_line_var=_plot_clustering_report__overlay_line_var,
    cluster_colors=_plot_clustering_report__cluster_colors,
    cluster_stats_sv_data_var=_plot_clustering_report__cluster_stats_sv_data_var,
)

*End of included section: processing_lvl_4.yaml*